### Notebook 1 — Exploration et prise en main de Spark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TradeCorp ETL").getOrCreate()

In [4]:
DATA_PATH = "/home/jovyan/data"   # adapte selon ton volume Docker

# Lecture des fichiers CSV
df_customers      = spark.read.csv(f"{DATA_PATH}/customers.csv", header=True, inferSchema=True)
df_orders         = spark.read.csv(f"{DATA_PATH}/orders.csv", header=True, inferSchema=True)
df_order_details  = spark.read.csv(f"{DATA_PATH}/order_details.csv", header=True, inferSchema=True)
df_products       = spark.read.csv(f"{DATA_PATH}/products.csv", header=True, inferSchema=True)
df_categories     = spark.read.csv(f"{DATA_PATH}/categories.csv", header=True, inferSchema=True)
df_suppliers      = spark.read.csv(f"{DATA_PATH}/suppliers.csv", header=True, inferSchema=True)
df_employers      = spark.read.csv(f"{DATA_PATH}/employees.csv", header=True, inferSchema=True)
df_shippers       = spark.read.csv(f"{DATA_PATH}/shippers.csv", header=True, inferSchema=True)

# Affichage des 5 premières lignes pour chaque DataFrame
dfs = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employers": df_employers,
    "shippers": df_shippers
}

In [75]:
#Lire les 8 fichiers
for name, df in dfs.items():
    print(f"\n=== {name.upper()} : 2 premières lignes ===")
    df.show(2)


=== CUSTOMERS : 2 premières lignes ===
+-----------+--------------------+------------+--------------------+--------------------+-----------+------+-----------+-------+------------+------------+
|customer_id|        company_name|contact_name|       contact_title|             address|       city|region|postal_code|country|       phone|         fax|
+-----------+--------------------+------------+--------------------+--------------------+-----------+------+-----------+-------+------------+------------+
|      ALFKI| Alfreds Futterkiste|Maria Anders|Sales Representative|       Obere Str. 57|     Berlin|  null|      12209|Germany| 030-0074321| 030-0076545|
|      ANATR|Ana Trujillo Empa...|Ana Trujillo|               Owner|Avda. de la Const...|México D.F.|  null|      05021| Mexico|(5) 555-4729|(5) 555-3745|
+-----------+--------------------+------------+--------------------+--------------------+-----------+------+-----------+-------+------------+------------+
only showing top 2 rows


=== 

In [76]:
# Tableau récapitulatif Python

summary = []

for name, df in dfs.items():
    count = df.count()
    summary.append((name, count))
    print(f"{name}: {count} lignes")

customers: 91 lignes
orders: 830 lignes
order_details: 2155 lignes
products: 77 lignes
categories: 8 lignes
suppliers: 29 lignes
employers: 9 lignes
shippers: 6 lignes


#### Statistiques descriptives

In [49]:
from pyspark.sql.functions import min, max, avg, stddev, lit

def compute_stats(df, name):
    print(f"\n===== Statistiques pour {name} =====")

    # Colonnes numériques uniquement
    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint")]

    if not numeric_cols:
        print("Aucune colonne numérique trouvée.")
        return None

    # Expressions de stats
    stats_exprs = []
    for c in numeric_cols:
        stats_exprs.extend(
            [
            min(c).alias(f"{c}_min"),
            max(c).alias(f"{c}_max"),
            avg(c).alias(f"{c}_mean"),
            stddev(c).alias(f"{c}_stddev")
        ])

    stats_df = df.select(*stats_exprs)
    stats_df = stats_df.withColumn("dataset", lit(name))

    stats_df.show(truncate=False)
    return stats_df


In [ ]:
#stats_orders = compute_stats(df_orders, "orders")
#stats_products = compute_stats(df_products, "products")

#### Comaparer Pandas vs Pysspark

In [59]:
#Pandasimport pandas as pd
import time

start = time.time()
pdf_orders = pd.read_csv("/home/jovyan/data/orders.csv")
end = time.time()

pandas_time = end - start
print(f"Temps Pandas : {pandas_time:.4f} secondes")


Temps Pandas : 0.0129 secondes


In [60]:
#Pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import time

spark = SparkSession.builder.getOrCreate()

start = time.time()
df_orders = spark.read.csv("/home/jovyan/data/orders.csv", header=True, inferSchema=True)
df_orders.count()   # force Spark à lire le fichier
end = time.time()

spark_time = end - start
print(f"Temps Spark : {spark_time:.4f} secondes")



Temps Spark : 0.3567 secondes


In [61]:
summary = [
    ("pandas", pandas_time),
    ("spark", spark_time)
]

summary_df = spark.createDataFrame(summary, ["methode", "temps_seconds"])
summary_df.show()


+-------+--------------------+
|methode|       temps_seconds|
+-------+--------------------+
| pandas|0.012947797775268555|
|  spark| 0.35665202140808105|
+-------+--------------------+



In [66]:


cols_to_cast = []

for col, dtype in df_orders.dtypes:
    # Colonnes numériques mal typées
    if dtype == "string":
        # Test : est-ce que la colonne contient uniquement des chiffres ?
        if df_orders.select(col).rdd.map(lambda x: x[0]).filter(lambda x: x is not None).filter(lambda x: not x.isdigit()).count() == 0:
            cols_to_cast.append((col, "int"))
        
        # Test : est-ce que la colonne ressemble à une date ?
        elif df_orders.select(col).rdd.map(lambda x: x[0]).filter(lambda x: x is not None).filter(lambda x: "-" in x).count() > 0:
            cols_to_cast.append((col, "date"))

print("\nColonnes qui nécessitent un cast :")
print(cols_to_cast)


Colonnes qui nécessitent un cast :
[('ship_name', 'date'), ('ship_address', 'date'), ('ship_postal_code', 'date')]


#### fin notebook 1